In [ ]:
import torch
from matplotlib import pyplot as plt

from src.components.machine_learning.loss_functions.gaussian_smoothed_cross_entropy_loss import GaussianSmoothedCrossEntropyLoss

In [ ]:
torch.set_printoptions(sci_mode=False)

In [ ]:
gaussian_loss = GaussianSmoothedCrossEntropyLoss(60, 0.5)

In [ ]:
distribution = gaussian_loss._make_soft_labels(torch.Tensor([10]))[0]

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(
    range(len(distribution)),
    distribution,
    color='black'
    )

plt.xlabel("Value")
plt.ylabel("Probability")
plt.title(f"Distribution")
plt.show()

In [ ]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

# PREDICTING: #

In [ ]:
from datetime import datetime 
from matplotlib import pyplot as plt
from dataclasses import dataclass
import math

from src.modules.model_managers.player_props import model_manager
from src.components.data_preparation import nba_data_helpers

In [ ]:
removed_core_teammates: set[int] = set([
    nba_data_helpers.player_name_to_ids('Austin Reaves')[0]
])

added_core_teammates: set[int] = set([
    # nba_data_helpers.player_name_to_ids('Luka Doncic')[0]
])

predicted_probabilities = model_manager.predict(
    model_manager.PLAYER_PROPS_POINTS_MODEL, 
    nba_data_helpers.player_name_to_ids('Luka Doncic')[0],
    nba_data_helpers.team_name_to_id('lakers'),
    nba_data_helpers.team_name_to_id('spurs'),
    datetime(2026, 6, 30),
    nba_data_helpers.SeasonTypes.PLAYOFFS,
    False,
    removed_core_teammates,
    added_core_teammates
)

In [ ]:
predicted_probabilities

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(
    range(len(predicted_probabilities)),
    predicted_probabilities,
    color='black'
    )

plt.xlabel("Value")
plt.ylabel("Probability")
plt.title(f"Predictions")
plt.show()

In [ ]:
expected_value = sum(class_index * probability for class_index, probability in enumerate(predicted_probabilities))
expected_value

In [ ]:
@dataclass(frozen=True)
class LineOdds():
    """
    What each way a line can settle is worth.

    A whole number line can be landed on exactly, which is a push. The stake comes back, so it is
    neither a win nor a loss and has to be priced on its own. Most published lines end in .5 and
    cannot be pushed, leaving push at zero.
    """

    over: float
    under: float
    push: float

def calculate_odds(
    predicted_probabilities: list[float],
    over_under_line: float,
    minimum_value: int = 0
    ) -> LineOdds:
    """
    Splits a predicted probability distribution into the chance of going over a line, under it, and
    landing exactly on it.

    Args:
        predicted_probabilities is a list of floats where the value at each index represents the predicted probabilty of
        the value represented by that index of occuring. Each value is assumed to be ordered and in increments of exactly 1.

        over_under_line is a threshold set that a given value has a chance of exceeding.

        minimum_value is the value that the index at 0 of the predicted_probability represents.
    """

    def clamped(index: float) -> int:
        return min(max(int(index), 0), len(predicted_probabilities))

    # Only a value strictly above the line goes over, so a whole number line settles the value equal
    # to it as a push rather than as a win.
    first_over_index = clamped(math.floor(over_under_line) + 1 - minimum_value)
    first_not_under_index = clamped(math.ceil(over_under_line) - minimum_value)

    return LineOdds(
        over=sum(predicted_probabilities[first_over_index:]),
        under=sum(predicted_probabilities[:first_not_under_index]),
        push=sum(predicted_probabilities[first_not_under_index:first_over_index])
    )

def calculate_odds_over_line(
    predicted_probabilities: list[float],
    over_under_line: float,
    minimum_value: int = 0
    ) -> float:
    """
    Calculates the probability given the probability distribution of a value to be higher than the
    given over_under_line. See calculate_odds for the arguments and for the other two outcomes.
    """

    return calculate_odds(predicted_probabilities, over_under_line, minimum_value).over

In [ ]:
odds = calculate_odds_over_line(
    predicted_probabilities,
    25.5
)
odds

: 